# DeepChem × AlphaFold: 예측 구조에 리간드 도킹 (v0.2.3)

이 노트북은 **"단백질 구조 → 도킹"** 파트를 담당합니다.
AlphaFold **구조 예측은 안정적인 [공식 ColabFold 노트북](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)** 에서 따로 수행하고, 그 결과 `.pdb`를 여기로 가져와 도킹합니다.

> 💡 ColabFold를 이 노트북에 설치하지 않으므로 **numpy/TensorFlow 충돌이 없습니다.** (DeepChem만 사용)

## 전체 워크플로우

1. **[공식 ColabFold 노트북](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)** 에서
   - 단백질 서열 입력 → 실행 → 예측 구조 **`.pdb` 다운로드** (rank_001 권장)
2. **이 노트북에서**
   - 그 `.pdb` 업로드 → 포켓 탐색 → 리간드 도킹 → 점수·시각화

> AlphaFold가 처음이거나 시간이 없으면, 업로드를 건너뛰고 **예시 구조(1M17)** 로 바로 실습할 수 있습니다.

## 환경 설정 (설치)

도킹에 필요한 DeepChem과 도구들을 설치합니다.

> ⚠️ 설치하면 numpy가 1.26으로 바뀌어 **"런타임 다시 시작"** 안내가 한 번 뜹니다. 재시작 후 이 셀은 건너뛰고 아래부터 이어서 실행하세요.

In [ ]:
!pip install -q --pre deepchem
!pip install -q openmm pdbfixer vina mdtraj py3Dmol rdkit
!mkdir -p dock_out

## STEP 1. 예측 구조(PDB) 준비

공식 ColabFold에서 받은 `.pdb`를 업로드하세요. (업로드를 취소하면 예시 구조 **1M17**을 자동으로 사용합니다.)

In [ ]:
import os
from google.colab import files

generated_pdb = None
print("예측한 .pdb 파일을 업로드하세요. (없으면 [취소] → 예시 1M17 사용)")
try:
    up = files.upload()
    if up:
        generated_pdb = list(up.keys())[0]
except Exception:
    pass

# 업로드가 없으면 예시 단백질(1M17, EGFR) 사용
if not generated_pdb or not os.path.exists(generated_pdb):
    import requests
    generated_pdb = "1M17.pdb"
    open(generated_pdb, "w").write(requests.get("https://files.rcsb.org/download/1M17.pdb").text)
    print("업로드가 없어 예시 구조(1M17)를 사용합니다.")

print("도킹 대상 단백질:", generated_pdb)

## STEP 2. 후보 리간드 입력

도킹할 리간드들의 **SMILES**를 넣으세요. ([PubChem](https://pubchem.ncbi.nlm.nih.gov/)에서 분자 이름으로 검색해 복사)

In [ ]:
# 후보 리간드 (자유롭게 추가/수정)
ligand_library = {
    "ibuprofen": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O",
    "aspirin":   "CC(=O)OC1=CC=CC=C1C(=O)O",
    "erlotinib_analog": "COC1=C(C=C2C(=C1)N=CN=C2NC3=CC(=C(C=C3)Cl)F)OCCCN4CCOCC4",
}
print("후보 리간드 수:", len(ligand_library))

## STEP 3. 포켓 탐색

`ConvexHullPocketFinder`로 단백질의 결합 포켓을 찾고, 첫 포켓 중심을 도킹 박스 중심으로 사용합니다.

In [ ]:
import numpy as np
import deepchem as dc
from deepchem.dock import ConvexHullPocketFinder

finder = ConvexHullPocketFinder()
pockets = finder.find_pockets(generated_pdb)
print("탐색된 포켓 수:", len(pockets))

pk = pockets[0]
centroid = np.array([(pk.x_range[0]+pk.x_range[1])/2,
                     (pk.y_range[0]+pk.y_range[1])/2,
                     (pk.z_range[0]+pk.z_range[1])/2])
print("도킹 중심(centroid):", centroid.round(2))

## STEP 4. 도킹 (리간드별 자세 생성 + 점수)

리간드마다 준비 → 도킹을 수행하고 **가장 좋은(가장 음수) 점수**를 모읍니다. (실패한 리간드는 자동 건너뜀)

In [ ]:
import os
from rdkit import Chem
from rdkit.Chem import AllChem
from deepchem.dock import VinaPoseGenerator
from deepchem.utils.vina_utils import prepare_inputs

os.makedirs("dock_out", exist_ok=True)
vpg = VinaPoseGenerator()

results = []
for name, smi in ligand_library.items():
    try:
        p, m = prepare_inputs(generated_pdb, smi)
    except Exception as e:
        print(f"[{name}] 준비 실패 → 건너뜀: {e}"); continue
    prot, lig = f"dock_out/prot_{name}.pdb", f"dock_out/lig_{name}.pdb"
    Chem.rdmolfiles.MolToPDBFile(p, prot)
    Chem.rdmolfiles.MolToPDBFile(m, lig)
    try:
        poses, scores = vpg.generate_poses(
            molecular_complex=(prot, lig),
            centroid=centroid, box_dims=np.array([20.0, 20.0, 20.0]),
            exhaustiveness=4, num_modes=5,
            out_dir="dock_out", generate_scores=True)
    except Exception as e:
        print(f"[{name}] 도킹 실패 → 건너뜀: {e}"); continue
    best = float(np.min(scores))
    results.append({"ligand": name, "best_score": best, "ligand_pose": poses[0][1]})
    print(f"[{name}] best score = {best:.2f} kcal/mol")

print("\n도킹 완료:", len(results), "개")

## STEP 5. 결과 순위표 + 막대그래프

점수가 **낮을수록(더 음수) 강한 결합**입니다.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rank = pd.DataFrame([{"ligand": r["ligand"], "best_score (kcal/mol)": round(r["best_score"], 2)}
                     for r in results]).sort_values("best_score (kcal/mol)").reset_index(drop=True)
rank.index += 1
rank.index.name = "rank"
display(rank)

order = rank.iloc[::-1]
plt.figure(figsize=(7, 4))
plt.barh(order["ligand"], order["best_score (kcal/mol)"], color="steelblue")
plt.xlabel("best docking score (kcal/mol, lower = stronger)")
plt.title("Ligand docking ranking")
plt.tight_layout()
plt.show()

## STEP 6. 1위 복합체 시각화

가장 점수가 좋은 리간드가 단백질 포켓에 어떻게 들어앉는지 봅니다.
회색 cartoon: 단백질 / 초록 stick: 1위 리간드 / 청록: 주변 5Å 잔기

In [ ]:
import py3Dmol

top = min(results, key=lambda r: r["best_score"])
print("🏆 1위:", top["ligand"], "| score:", round(top["best_score"], 2), "kcal/mol")

with open(generated_pdb) as f:
    prot_block = f.read()
lig_block = Chem.MolToMolBlock(top["ligand_pose"])

view = py3Dmol.view(width=800, height=500)
view.addModel(prot_block, "pdb")
view.setStyle({"model": 0}, {"cartoon": {"color": "lightgray"}})
view.addModel(lig_block, "mol")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.3}})
view.addStyle({"model": 0, "within": {"distance": 5, "sel": {"model": 1}}},
              {"stick": {"colorscheme": "cyanCarbon", "radius": 0.2}})
view.zoomTo({"model": 1})
view.show()

## 마무리

- 도킹 점수(kcal/mol)는 **음수일수록 강한 결합**, 더 음수일수록 안정적입니다.
- 여기서는 **AlphaFold가 예측한(또는 실험) 구조**에 리간드를 도킹해 친화도를 비교했습니다.
- 점수의 절대값보다 **리간드 간 상대 순위·결합 부위 위치**를 해석하는 데 집중하세요.

> 구조 예측이 궁금하면 [공식 ColabFold 노트북](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb)에서 직접 서열을 예측해 그 PDB를 STEP 1에 올려보세요.